### Library

In [12]:
import os
import glob
import fitz  # pymupdf
import faiss
import numpy as np
import pickle
from langchain_community.embeddings import HuggingFaceBgeEmbeddings
from dotenv import load_dotenv
from tqdm.auto import tqdm


### .env and directory

In [13]:
# Load environment variables from .env
load_dotenv()

# Directories for your project
BASE_DIR = os.getcwd()
PDF_DIR = os.path.join(BASE_DIR, 'pdfs')
DATA_DIR = os.path.join(BASE_DIR, 'rag_data')
INDEX_DIR = os.path.join(DATA_DIR, 'faiss_index')
META_PATH = os.path.join(DATA_DIR, 'metadata.pkl')
os.makedirs(PDF_DIR, exist_ok=True)
os.makedirs(INDEX_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)


BAAI model

In [14]:
# Load the BAAI embedding model
model_name = "BAAI/bge-base-en-v1.5"
encode_kwargs = {'normalize_embeddings': True}
embeddings_model = HuggingFaceBgeEmbeddings(
    model_name=model_name, model_kwargs={'device': 'cpu'}, encode_kwargs=encode_kwargs
)


### Pdf to text

In [15]:
def extract_text_from_pdf(path: str) -> str:
    """Extract plain text from a PDF file using PyMuPDF (fitz)."""
    text_parts = []
    doc = fitz.open(path)
    for page_no in range(len(doc)):
        page = doc.load_page(page_no)
        page_text = page.get_text("text")
        if page_text:
            page_text = "\n".join([line.strip() for line in page_text.splitlines() if line.strip()])
            text_parts.append(page_text)
    doc.close()
    return "\n\n".join(text_parts)

pdf_paths = sorted(glob.glob(os.path.join(PDF_DIR, "*.pdf")))
all_docs = []
for p in tqdm(pdf_paths, desc='Reading PDFs'):
    txt = extract_text_from_pdf(p)
    doc = {
        'id': os.path.basename(p),
        'text': txt,
        'source': p
    }
    all_docs.append(doc)


Reading PDFs:   0%|          | 0/16 [00:00<?, ?it/s]

### Chunking

In [16]:
# from typing import List

# def chunk_text(text: str, chunk_size: int = 800, overlap: int = 200) -> List[str]:
#     if not text:
#         return []
#     tokens = text.split()
#     avg_word_len = max(1, sum(len(w) for w in tokens) / len(tokens))  # Average word length
#     words_per_chunk = max(50, int(chunk_size / avg_word_len))  # Number of words per chunk
#     overlap_words = max(10, int(overlap / avg_word_len))  # Number of words to overlap

#     chunks = []
#     start = 0
#     while start < len(tokens):
#         end = min(len(tokens), start + words_per_chunk)
#         chunk = " ".join(tokens[start:end])
#         chunks.append(chunk)
#         if end == len(tokens):
#             break
#         start = end - overlap_words
#     return chunks


In [17]:
# chunked_docs = []
# for doc in all_docs:  
#     chunks = chunk_text(doc['text'])
#     for i, c in enumerate(chunks):
#         chunked_docs.append({
#             'chunk_id': f"{doc['id']}_chunk_{i}",
#             'text': c,
#             'source': doc['id'],
#             'chunk_index': i
#         })


### semantic chunking

In [18]:
from typing import List
import re

def chunk_text_by_topics(text: str, min_chunk_size: int = 800) -> List[str]:
    """
    Chunk the text based on semantic topics (identified via headings or sections).
    This is a topic-based chunking approach.
    
    :param text: The full text to be chunked.
    :param min_chunk_size: The minimum size of each chunk.
    :return: A list of topic-based chunks.
    """
    
    if not text:
        return []
    
    # Regular expression to match headings (e.g., "Section 1: CPF Contributions")
    heading_pattern = r'(?:^|\n)([A-Z][^:\n]+:.*?)(?=\n|$)'  # Matches headings (e.g., Section 1: ... or 1. ... )
    
    headings = re.findall(heading_pattern, text)
    
    if not headings:
        # If no headings, we could default to chunking by a simple length rule (e.g., same as your original method)
        return [text[i:i+min_chunk_size] for i in range(0, len(text), min_chunk_size)]
    
    chunks = []
    start_index = 0
    for heading in headings:
        # Find the position of the heading in the text
        heading_position = text.find(heading, start_index)
        
        # If there's content before the heading, chunk it as a separate piece
        if heading_position > start_index:
            chunk = text[start_index:heading_position].strip()
            if len(chunk) >= min_chunk_size:
                chunks.append(chunk)
        
        # Find the content for the topic section
        next_heading_position = text.find(headings[headings.index(heading) + 1], heading_position) if headings.index(heading) + 1 < len(headings) else len(text)
        topic_chunk = text[heading_position:next_heading_position].strip()
        
        # Ensure each chunk has a meaningful size (minimum chunk size rule)
        if len(topic_chunk) >= min_chunk_size:
            chunks.append(topic_chunk)
        
        # Move to the next section
        start_index = next_heading_position
    
    return chunks


# Assuming all_docs contains the documents in a format like {'id': ..., 'text': ...}
chunked_docs = []
for doc in all_docs:
    chunks = chunk_text_by_topics(doc['text'])
    for i, c in enumerate(chunks):
        chunked_docs.append({
            'chunk_id': f"{doc['id']}_chunk_{i}",
            'text': c,
            'source': doc['id'],
            'chunk_index': i
        })


### Embedding

In [19]:
texts = [c['text'] for c in chunked_docs]  
embeddings = embeddings_model.embed_documents(texts)  
emb_matrix = np.array(embeddings, dtype='float32')  
faiss.normalize_L2(emb_matrix)  


### Save Vector database

In [20]:
# Create the FAISS index using Inner Product (IP)
index = faiss.IndexFlatIP(emb_matrix.shape[1])

# Add the embeddings to the index
index.add(emb_matrix)

# Save the FAISS index to disk
faiss.write_index(index, os.path.join(INDEX_DIR, 'faiss.index'))


### Save metadata

In [21]:
with open(META_PATH, 'wb') as f:
    pickle.dump(chunked_docs, f)


### save embedding file

In [22]:
# Save embeddings to a file (optional, to load later in Streamlit)
with open('embeddings.pkl', 'wb') as f:
    pickle.dump(emb_matrix, f)
